# Checkpoint analysis

Analyze Walrus checkpoints straight from Weights & Biases runs. Given a project and run
name (or id), the helpers locate the checkpoint that minimizes a validation metric,
reload the model, and run the same **test** (one-step) and **rollout_test** evaluation
used during training.

Two entry points (from `walrus.analysis`):

- **`show_run(project, run, metric="rollout_valid")`** — for a single run, evaluates the
  checkpoint best by rollout ``VRMSE`` by default (`metric="rollout_valid"`); pass
  `metric="valid"` for one-step, or `metric="both"` for both. Prints test / rollout_test
  **VRMSE** scores and shows rollout videos.
- **`compare_runs(project, [run, ...])`** — table of test / rollout_test **VRMSE** for the
  best-by-rollout-VRMSE and best-by-one-step-VRMSE checkpoint of every run.
  (Not the train `loss_fn` scalar — that is MAE vs CRPS across runs.)
- **`compare_zero_shot(project, [run, ...], data=...)`** — table of zero-shot
  **rollout_test VRMSE** on another Well dataset (best-by-rollout checkpoint; no original-test re-eval).

Checkpoints are selected from the epochs that were actually saved on disk (`step_*`,
`best`, `last`), so the reported metric always matches a real checkpoint.


In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Make the walrus package importable regardless of the notebook's working directory.
import pathlib
import sys

_cwd = pathlib.Path.cwd().resolve()
REPO_ROOT = next(
    p for p in (_cwd, *_cwd.parents) if (p / "walrus" / "__init__.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from walrus.analysis import (
    compare_runs,
    compare_zero_shot,
    evaluate_flow_metrics,
    evaluate_t0_omega,
    get_run,
    plot_all_autocorrelations,
    plot_rms_velocity_overlay,
    run_history,
    select_checkpoint,
    show_run,
    t0_omega_from_well,
)

# Wandb project. entity=None uses your default wandb entity.
#
# Old 128x128 morphogenesis runs:
#   PROJECT = "morphogenesis"
#   path=.../processed_morphodynamic_atlas remaps to WT_old automatically.
#   A few July 22-23 runs already say path=.../WT (when that folder was still 128).
#   For those, set DATA = "morphogenesis_WT_old".
# Current 64x96 / myosin: PROJECT = "morphogenesis_myosin", DATA = None.
PROJECT = "morphogenesis_myosin"
ENTITY = None
DATA = None  #"morphogenesis_WT_old"  # or "morphogenesis_WT_old" for path=.../WT era runs
# Hydra data config for compare_zero_shot (different Well tree than training).
ZERO_SHOT_DATA = "morphogenesis_WT_myosin"

# Set only if checkpoints were moved from the path recorded in the run config
# (e.g. trained on another machine). Otherwise leave as None.
WELL_BASE_PATH = None

# Single-run analyses (peek / show_run / t0_omega / flow metrics).
RUN = "Walrus_crps_morph_myosin_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001"

# Multi-run comparison / VRMSE overlay.
RUNS = [
    "Walrus_ft_morpho_n_steps-10-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001",
    "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001",
]


In [ ]:
import wandb
from walrus.analysis.checkpoint_analysis import available_checkpoints
import pathlib

api = wandb.Api()
entity = ENTITY or api.default_entity
wb_runs = api.runs(f"{entity}/{PROJECT}")

# Finished runs that actually saved a checkpoint. Use wandb ids so duplicate
# display names stay distinct (get_run(name) would keep only the newest).
wb_runs = [
    r
    for r in wb_runs
    if r.state == "finished"
    and available_checkpoints(pathlib.Path(r.config["checkpoint"]["save_dir"]))
]
wb_runs = sorted(wb_runs, key=lambda r: r.name)
RUNS = [r.id for r in wb_runs]
print(len(RUNS), "runs")
for r in wb_runs:
    print(f"{r.id}  {r.name}")

## Peek at what will be selected (cheap, no evaluation)

Check which checkpoint each criterion resolves to before running the (expensive)
evaluation. `run_history` returns per-epoch **VRMSE** (`valid` / `rollout_valid` columns
are mean `*/full_VRMSE_T=all_mean`).


In [ ]:
run = get_run(PROJECT, RUN, entity=ENTITY)

for metric in ("rollout_valid", "valid"):
    s = select_checkpoint(run, metric)
    print(f"best by {metric:>13} VRMSE: epoch {s.epoch:>4}  (VRMSE={s.value:.4f})  ->  {s.path}")

run_history(run).tail()


## Verify WT_old matches an old morphogenesis run

Compares a saved training `rollout_valid` VRMSE (from the run's `viz/loss_dicts`) to a
fresh eval on **WT_old**. Close agreement supports that WT_old is the right 128 data
(and that path remapping / stats work). Large disagreement means wrong folder,
wrong split layout, or regenerated stats differing from the lost original `stats.yaml`.

Also prints HDF5 `dataset_name` (training logged `drosophila_wt_sqh_mcherry_piv`).


In [ ]:
from walrus.analysis.checkpoint_analysis import load_config, evaluate_checkpoint
import h5py, glob, pickle, torch

run = get_run(PROJECT, RUN, entity=ENTITY)
cfg = load_config(run)
if DATA is not None:
    from walrus.analysis.checkpoint_analysis import apply_data_config
    cfg = apply_data_config(cfg, DATA)

info = cfg.data.module_parameters.well_dataset_info
path = next(iter(info.values())).path
norm = next(iter(info.values())).get("normalization_path", None)
print("resolved data path:", path)
print("normalization_path:", norm)

# Metadata check
f0 = sorted(glob.glob(f"{path}/data/train/*.hdf5"))[0]
with h5py.File(f0) as f:
    print("sample file:", f0)
    print("  dataset_name attr:", f.attrs.get("dataset_name"))
    print("  velocity shape:", f["t1_fields/velocity"].shape, "(expect T,x,y ~ 128x128)")

# Local training metric at a saved checkpoint epoch (no wandb API needed)
sel = select_checkpoint(run, "rollout_valid")
ep = sel.epoch
loss_pkl = Path(run.config["checkpoint"]["save_dir"]).parent / "viz" / "loss_dicts" / f"rollout_valid_loss_dict_epoch{ep}_rank0.pkl"
print("checkpoint epoch:", ep, "path:", sel.path)
print("training loss_dict:", loss_pkl, "exists:", loss_pkl.exists())

if loss_pkl.exists():
    _orig = torch.load
    torch.load = lambda *a, **k: _orig(*a, **{**k, "map_location": "cpu", "weights_only": False})
    d = pickle.load(open(loss_pkl, "rb"))
    torch.load = _orig
    key = next(k for k in d if str(k).endswith("full_VRMSE_T=all_mean") and "rollout_valid" in str(k))
    train_vrmse = float(d[key])
    print(f"logged rollout_valid full_VRMSE_T=all_mean @ epoch {ep}: {train_vrmse:.6f}")
    print("Re-run evaluate_checkpoint(..., full=True) on GPU and compare rollout_test/valid VRMSE to this value.")
    print("Note: analyze helpers eval the *test* split by default; for a strict match, compare against the same split used in training (valid).")


## Display a single run

Loads the best-by-`rollout_valid` and best-by-`valid` checkpoints, runs test +
rollout_test on each, prints the scores, and shows the rollout videos.

Pass `full=True` for the complete test set (slow). `max_rollout_steps=...` shortens
rollouts for a quick look.

In [ ]:
results = show_run(
    PROJECT,
    RUN,
    metric="rollout_valid",
    entity=ENTITY,
    well_base_path=WELL_BASE_PATH,
    data=DATA,
    full=False,
    make_videos=True,
    validation_ensemble_size=4,
    num_detailed_logs=4,
)


## Compare runs

One row per run; columns are `test_VRMSE` and `rollout_test_VRMSE` for the best-by-rollout
and best-by-single-step checkpoint. This re-runs evaluation per selected checkpoint, so it
can take a while with `full=True`.


In [ ]:
table = compare_runs(
    PROJECT,
    RUNS,
    entity=ENTITY,
    well_base_path=WELL_BASE_PATH,
    data=DATA,
    full=True,
)
table.round(4)


## Compare zero-shot across runs

For each id in `RUNS`, load the checkpoint with the lowest rollout-validation **VRMSE**,
evaluate **only** open-loop `rollout_test` on `ZERO_SHOT_DATA`, and collect a table.
Does not re-run the original training test set. Runs with no checkpoints are skipped.

In [ ]:
zs_table = compare_zero_shot(
    PROJECT,
    RUNS,
    data=ZERO_SHOT_DATA,
    entity=ENTITY,
    well_base_path=WELL_BASE_PATH,
    full=True,
    make_videos=False,
)
zs_table.round(4)


## Compare-runs bar chart

Grouped bars of **one-step test VRMSE** vs **open-loop rollout-test VRMSE** for each
checkpoint-selection rule. Epochs are omitted. Lower is better.

In [ ]:
from pathlib import Path

import matplotlib as mpl
from matplotlib.patches import Patch

# Embed TrueType (Type 42) so Illustrator / Inkscape can edit text as fonts.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
mpl.rcParams["hatch.linewidth"] = 1.1

SHORT_NAMES = {
    "FFNO_morph_WT-morph-delta-FFNOW[--]-AdamW-0.0001": "FFNO",
    "PoseidonL_ft_morph_WT-morph-full-ScOTW[--]-AdamW-0.0001": "Poseidon-L",
    "SineNet_morph_WT-morph-delta-SineN[--]-AdamW-0.0001": "SineNet",
    "Walrus_crps_morph_latent_noise-every-2-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus CRPS",
    "Walrus_ft_morph_WT_no_myosin-morph-delta-Isotr[Space-Adapt-]-AdamW-0.0001": "Walrus (det.)",
}

# Non-learned baselines have no checkpoints, so compare_runs cannot score them.
# Values are from their validation_mode test / rollout_test loss dicts (one eval).
extra = pd.DataFrame(
    {
        ("best_by_rollout", "test_VRMSE"): [0.6752, 0.6633],
        ("best_by_rollout", "rollout_test_VRMSE"): [1.1717, 1.3601],
        ("best_by_single_step", "test_VRMSE"): [0.6752, 0.6633],
        ("best_by_single_step", "rollout_test_VRMSE"): [1.1717, 1.3601],
    },
    index=["Advection", "Mean-field"],
)
table_plot = pd.concat([table, extra])

names = [SHORT_NAMES.get(i, i) for i in table_plot.index]
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2", "#FF9DA6"]
x = np.arange(len(names))
w = 0.36

fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.0), sharey=True)
panels = [
    (axes[0], "test_VRMSE", "One-step test VRMSE"),
    (axes[1], "rollout_test_VRMSE", "Open-loop rollout-test VRMSE"),
]
for ax, metric, title in panels:
    y_roll = table_plot[("best_by_rollout", metric)].to_numpy(dtype=float)
    y_step = table_plot[("best_by_single_step", metric)].to_numpy(dtype=float)
    bars_roll = ax.bar(
        x - w / 2, y_roll, w, color=colors[: len(names)],
        edgecolor="0.15", linewidth=0.6, zorder=3,
    )
    bars_step = ax.bar(
        x + w / 2, y_step, w, color=colors[: len(names)],
        edgecolor="0.15", linewidth=0.6, hatch="///", zorder=3,
    )
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xticks(x, names, rotation=25, ha="right")
    ax.set_xlabel("Model")
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for bars in (bars_roll, bars_step):
        for rect in bars:
            h = rect.get_height()
            ax.annotate(
                f"{h:.2f}",
                xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=7.5, color="0.25",
            )

ymax = float(np.nanmax(table_plot[[("best_by_rollout", "rollout_test_VRMSE"), ("best_by_single_step", "rollout_test_VRMSE")]].to_numpy(dtype=float)))
axes[0].set_ylim(0, ymax * 1.12)
axes[0].set_ylabel("VRMSE  (lower is better)", fontsize=12)

style_handles = [
    Patch(facecolor="0.75", edgecolor="0.15", label="Selected by rollout validation (solid)"),
    Patch(facecolor="0.75", edgecolor="0.15", hatch="///", label="Selected by one-step validation (hatched)"),
]
model_handles = [Patch(facecolor=c, edgecolor="0.15", label=n) for n, c in zip(names, colors)]
leg1 = axes[1].legend(handles=style_handles, loc="upper left", frameon=False, fontsize=9, title="Checkpoint selection")
axes[1].add_artist(leg1)
axes[1].legend(handles=model_handles, loc="upper left", bbox_to_anchor=(0.0, 0.72), frameon=False, fontsize=9, title="Model")

fig.suptitle("In-distribution VRMSE on morphogenesis WT (no myosin)", y=1.03)
fig.tight_layout()

out = Path("_analysis_viz") / "compare_runs_vrmse_no_myosin.pdf"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, bbox_inches="tight")
print(out.resolve())
plt.show()


## Visualize rollout VRMSE over time

Raw prediction videos hide small L2 gains. These curves are the same VRMSE that
feeds the table above, broken out by rollout lead time (from `_analysis_viz/.../rollout_losses/`).
Overlaying them usually makes the CRPS vs det gap obvious even when movies look similar.

Uses the `RUNS` list from the setup cell at the top and the **best-by-`rollout_valid`** eval folder.


In [ ]:
VIZ = Path("_analysis_viz")
SELECTION = "rollout_valid"  # best-by-rollout_valid eval folder
METRIC = "full_VRMSE_rollout_mean.npy"
FIELD_METRICS = [
    "velocity_x_VRMSE_rollout_mean.npy",
    "velocity_y_VRMSE_rollout_mean.npy",
]
LABELS = {
    RUNS[0]: "no crps",
    RUNS[1]: "crps",
} if len(RUNS) >= 2 else {}


def _label(run: str) -> str:
    return LABELS.get(run, run if len(run) <= 28 else run[:27] + "…")


def find_loss_dirs(run: str, selection: str = SELECTION):
    root = VIZ / run / selection
    if not root.exists():
        print(f"missing: {root}")
        return
    for loss_root in sorted(root.glob("*/rollout_losses")):
        dset = loss_root.parent.name
        dirs = sorted(loss_root.glob("*rollout_test*")) or sorted(loss_root.iterdir())
        for d in dirs:
            if d.is_dir():
                yield dset, d


curves = {}  # (run, dset, metric) -> array
for run in RUNS:
    for dset, loss_dir in find_loss_dirs(run):
        for metric in [METRIC, *FIELD_METRICS]:
            path = loss_dir / metric
            if path.exists():
                curves[(run, dset, metric)] = np.load(path)

if not curves:
    raise FileNotFoundError(
        f"No {METRIC} under {VIZ}/<run>/{SELECTION}/.../rollout_losses/. "
        "Run compare_runs(..., full=True) first."
    )

dsets = sorted({dset for (_, dset, _) in curves})


def plot_metric(metric: str, title: str):
    fig, axes = plt.subplots(
        1, len(dsets), figsize=(6.5 * len(dsets), 4), squeeze=False, sharey=True
    )
    for ax, dset in zip(axes[0], dsets):
        series = [
            (run, curves[(run, dset, metric)])
            for run in RUNS
            if (run, dset, metric) in curves
        ]
        if not series:
            ax.set_title(f"{dset} (missing)")
            continue
        # Align on shared horizon so length mismatch doesn't fake a gap.
        n = min(len(y) for _, y in series)
        for run, y in series:
            ax.plot(np.arange(n), y[:n], label=_label(run), linewidth=2)
            # faint full curve if longer than shared horizon
            if len(y) > n:
                ax.plot(
                    np.arange(n, len(y)),
                    y[n:],
                    color=ax.lines[-1].get_color(),
                    alpha=0.35,
                    linewidth=1.5,
                    linestyle="--",
                )
        ax.set_title(dset)
        ax.set_xlabel("rollout step")
        ax.set_ylabel("VRMSE")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)
        ax.axvline(n - 1, color="gray", alpha=0.4, linestyle=":", label=None)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_metric(METRIC, f"Rollout VRMSE vs lead time ({SELECTION}; solid = shared horizon)")
for field_metric in FIELD_METRICS:
    if any(m == field_metric for (_, _, m) in curves):
        field = field_metric.replace("_VRMSE_rollout_mean.npy", "")
        plot_metric(field_metric, f"{field} VRMSE vs lead time")

print("mean VRMSE over shared horizon (fair compare):")
for dset in dsets:
    series = [
        (run, curves[(run, dset, METRIC)])
        for run in RUNS
        if (run, dset, METRIC) in curves
    ]
    if len(series) < 2:
        continue
    n = min(len(y) for _, y in series)
    for run, y in series:
        print(
            f"  {dset} | {_label(run):8s}  len={len(y):3d}  "
            f"mean[:{n}]={y[:n].mean():.4f}  last_shared={y[n-1]:.4f}  "
            f"full_mean={y.mean():.4f}"
        )
    print(
        "  note: table rollout_test can look much worse for det if its trajectories "
        "are longer (error accumulates past the shared horizon)."
    )


## $t_0^\omega$ landmark (GT vs predictions)

Mitchell et al. Nat Methods 2026: VF→GBE time from the vorticity autocorrelation.

- **GT file**: `t0_omega` on the Well velocity field (absolute developmental time).
- **GT rollout / pred**: same landmark on the stitched trajectory
  `concat(context, y_ref)` vs `concat(context, y_pred)` from the best-by-`rollout_valid`
  checkpoint. Context is ground truth; predictions are open-loop after it.

Needs a GPU. Uses absolute `t` / `dx` / `dy` from the Well file (batch time grids are normalized).


In [ ]:
# Cheap: GT-only on the Well test files (no model).
# For old morphogenesis / WT_old: use the 128 test split.
test_dir = Path("/data/lcornelis/morphogenesis_data/WT_old/data/test")
gt_rows = [t0_omega_from_well(p) for p in sorted(test_dir.glob("*.hdf5"))]
gt_table = pd.DataFrame(gt_rows)
print("Well-file GT t0_omega (expect near 0):")
display(gt_table[["path", "t0v_frame", "t0_omega", "n_frames"]])

# Model: GT-stitched vs open-loop pred on rollout_test trajectories.
t0_df = evaluate_t0_omega(
    get_run(PROJECT, RUN, entity=ENTITY),
    metric="rollout_valid",
    split="rollout_test",
    well_base_path=WELL_BASE_PATH,
    data=DATA,
    max_rollout_steps=200,  # cover full embryo trajectories
)
display(t0_df)
print(
    f"mean |t0_pred - t0_gt| = {t0_df['abs_err'].mean():.3f}   "
    f"(n={len(t0_df)})"
)


## Flow autocorrelation + r.m.s. velocity (Fig. 3c / 3i)

Mitchell et al. Nat Methods 2026 — same definitions as the paper / SI:

- **Autocorrelation** (Fig. 3c): vorticity Pearson correlation matrix
  (SI Note 10 Eq. 8). Caption: *Flow correlations: vorticity method*.
- **r.m.s. tissue velocity** (Fig. 3i): spatially averaged
  \(v_{\mathrm{RMS}}=\sqrt{\frac{1}{A}\iint |v|^2\,dA}\) (SI Eq. 9 / Note 13),
  in µm min⁻¹. Equal-area grid weights (Well pullbacks are already AP-cropped).

For each test embryo we compare **true** `concat(context, y_ref)` vs **predicted**
`concat(context, y_pred)` from the best-by-`rollout_valid` checkpoint.

**Time 0 = onset of GBE**, defined as in the paper: *"the time when the derivative
of the root-mean-squared velocity is maximal"* (chosen there for consistency with
earlier publications). This is not the \(t_0^\omega\) vorticity landmark, which is
still reported per embryo for reference.

Needs a GPU. Reuses the same rollout stitching as the \(t_0^\omega\) cell above.


In [ ]:
flow_results = evaluate_flow_metrics(
    get_run(PROJECT, RUN, entity=ENTITY),
    metric="rollout_valid",
    split="rollout_test",
    well_base_path=WELL_BASE_PATH,
    data=DATA,
    max_rollout_steps=200,
)
print(f"embryos: {len(flow_results)}")
for r in flow_results:
    print(
        f"  {r.file or r.batch}: GBE onset gt={r.gbe_onset_gt:.2f} / "
        f"pred={r.gbe_onset_pred:.2f} min (t0_omega_gt={r.t0_omega_gt:.2f}), "
        f"n={len(r.t)}, rms_gt peak={r.rms_gt.max():.2f}, "
        f"rms_pred peak={r.rms_pred.max():.2f}"
    )

# Fig. 3i-style: ensemble mean ± s.d. true vs predicted RMS velocity
fig, ax = plt.subplots(figsize=(5.5, 3.5))
plot_rms_velocity_overlay(flow_results, ax=ax)
plt.show()

# Fig. 3c-style: per-embryo true vs predicted autocorrelation heatmaps
plot_all_autocorrelations(flow_results)
